# Preprocessing Pipelines and Hyperparameter Tuning

Real datasets contain missing values, numeric columns on different scales, and categorical columns.
A scikit-learn `Pipeline` bundles preprocessing and the model into a single object, which:

- prevents data leakage during cross-validation,
- makes experiments reproducible,
- allows hyperparameter tuning of preprocessing steps and the model together,
- simplifies deployment (one object to save and load).

This notebook builds a pipeline for a mixed-type dataset, then tunes it with `GridSearchCV` and
`RandomizedSearchCV`.

## Learning objectives

1. Build a `ColumnTransformer` for numeric and categorical columns.
2. Combine preprocessing and a model in a `Pipeline`.
3. Tune an entire pipeline with `GridSearchCV` and `RandomizedSearchCV`.
4. Understand why nested cross-validation is needed for an unbiased performance estimate.
5. Avoid the most common leakage mistakes.

## 1. Concepts

### Pipeline

`Pipeline(steps=[("preprocess", transformer), ("model", estimator)])` calls `fit_transform` on all
steps except the last during training, and `transform` on all steps before predicting.

### ColumnTransformer

Different columns need different preprocessing:

- numeric: impute missing values, then scale;
- categorical: impute missing values, then one-hot encode.

`ColumnTransformer` applies each transformer to a subset of columns and concatenates the results.

### Grid search vs random search

- **Grid search** exhaustively evaluates every combination. Good for small, focused search spaces.
- **Random search** samples combinations. Often finds a good configuration faster when many
  hyperparameters have low impact.
- Both use cross-validation internally. The best score is an estimate on the validation folds, not
  an unbiased estimate of final performance.

### Nested cross-validation

If you tune hyperparameters and report the best cross-validation score, the score is optimistically
biased. Nested CV uses an outer loop for evaluation and an inner loop for tuning, giving a cleaner
estimate at extra computational cost.

## 2. Setup

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

## 3. Build a mixed-type dataset

We generate a synthetic classification problem and add two categorical columns and some missing
values. This mimics a realistic tabular workflow without relying on an external download.

In [2]:
X_num, y = make_classification(
    n_samples=1200,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    weights=[0.7, 0.3],
    class_sep=0.9,
    random_state=RANDOM_STATE,
)

numeric = pd.DataFrame(
    X_num,
    columns=[f"num_{i}" for i in range(X_num.shape[1])],
)

rng = np.random.default_rng(RANDOM_STATE)
categorical = pd.DataFrame({
    "contract": rng.choice(["monthly", "one_year", "two_year"], size=len(numeric), p=[0.5, 0.3, 0.2]),
    "region": rng.choice(["north", "south", "east", "west"], size=len(numeric)),
})

df = pd.concat([numeric, categorical], axis=1)
df["target"] = y

# Introduce realistic missing values.
for column in ["num_0", "num_2"]:
    mask = rng.random(len(df)) < 0.08
    df.loc[mask, column] = np.nan
df.loc[rng.random(len(df)) < 0.05, "region"] = np.nan

df.head()

,num_0,num_1,num_2,num_3,num_4,num_5,contract,region,target
0,-1.344245,0.255090,0.348329,-0.192606,2.374793,1.497609,one_year,north,0
1,-1.522165,0.605169,-1.067604,-1.981428,-0.855460,-0.880108,monthly,east,0
2,-0.239577,0.299652,-2.088066,1.719982,2.443740,1.546877,two_year,east,1
3,0.325372,0.948509,-1.333386,-0.489259,-1.609553,-0.940646,one_year,east,0
4,-0.758251,-0.103807,-0.126748,0.001949,-0.281111,-1.306544,monthly,south,0


In [3]:
print(df.isna().sum())
print(df.dtypes)

num_0       102
num_1         0
num_2        93
num_3         0
num_4         0
num_5         0
contract      0
region       64
target        0
dtype: int64
num_0       float64
num_1       float64
num_2       float64
num_3       float64
num_4       float64
num_5       float64
contract     object
region       object
target        int32
dtype: object


## 4. Train/test split

In [4]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (960, 8) Test: (240, 8)


## 5. Build the preprocessing pipeline

The `ColumnTransformer` handles numeric and categorical columns differently. The classifier is the
final step, so the whole object can be tuned and deployed as one unit.

In [5]:
numeric_features = numeric.columns.tolist()
categorical_features = categorical.columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['num_0', 'num_1', 'num_2',
                                                   'num_3', 'num_4', 'num_5']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['contract', 'region'])])),
                ('model', LogisticRegression(max_iter=5000, random_state=42))])

### Baseline cross-validation

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"Baseline ROC-AUC: {baseline_scores.mean():.4f} (+/- {baseline_scores.std():.4f})")

Baseline ROC-AUC: 0.7886 (+/- 0.0527)


## 6. Grid search over preprocessing and model hyperparameters

Prefixes in the parameter grid refer to pipeline steps: `model__C` means the `C` argument of the
model step; `preprocess__numeric__imputer__strategy` would target the numeric imputer.

In [7]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"],
    "preprocess__numeric__imputer__strategy": ["mean", "median"],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True,
)

start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start

print(f"Grid search time: {grid_time:.2f}s")
print("Best ROC-AUC:", round(grid_search.best_score_, 4))
print("Best parameters:", grid_search.best_params_)

Grid search time: 2.40s
Best ROC-AUC: 0.7953
Best parameters: {'model__C': 0.01, 'model__class_weight': 'balanced', 'preprocess__numeric__imputer__strategy': 'median'}


In [8]:
results = pd.DataFrame(grid_search.cv_results_)
results[[
    "param_model__C",
    "param_model__class_weight",
    "param_preprocess__numeric__imputer__strategy",
    "mean_test_score",
    "mean_train_score",
]].sort_values("mean_test_score", ascending=False).head(10).round(4)

,param_model__C,param_model__class_weight,param_preprocess__numeric__imputer__strategy,mean_test_score,mean_train_score
3,0.01,balanced,median,0.7953,0.7994
2,0.01,balanced,mean,0.7952,0.7994
0,0.01,None,mean,0.7940,0.7970
1,0.01,None,median,0.7939,0.7969
7,0.10,balanced,median,0.7934,0.8018
6,0.10,balanced,mean,0.7933,0.8018
11,1.00,balanced,median,0.7918,0.8020
10,1.00,balanced,mean,0.7917,0.8019
5,0.10,None,median,0.7911,0.7986
4,0.10,None,mean,0.7910,0.7986


## 7. Random search

Random search samples a fixed number of combinations. It is often more efficient when the search
space is large or when only a few hyperparameters matter.

In [9]:
from scipy.stats import loguniform

param_distributions = {
    "model__C": loguniform(1e-3, 1e2),
    "model__class_weight": [None, "balanced"],
    "preprocess__numeric__imputer__strategy": ["mean", "median"],
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start

print(f"Random search time: {random_time:.2f}s")
print("Best ROC-AUC:", round(random_search.best_score_, 4))
print("Best parameters:", random_search.best_params_)

Random search time: 0.48s
Best ROC-AUC: 0.7948
Best parameters: {'model__C': 0.028888383623653185, 'model__class_weight': 'balanced', 'preprocess__numeric__imputer__strategy': 'mean'}


## 8. Final check on the held-out test set

The test set has not been used for tuning. We evaluate the selected model once to estimate how it
might perform on unseen data.

In [10]:
best_model = grid_search.best_estimator_
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = best_model.predict(X_test)
from sklearn.metrics import roc_auc_score

print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print(classification_report(y_test, test_pred))

Test ROC-AUC: 0.7486
              precision    recall  f1-score   support

           0       0.83      0.82      0.82       168
           1       0.58      0.60      0.59        72

    accuracy                           0.75       240
   macro avg       0.70      0.71      0.70       240
weighted avg       0.75      0.75      0.75       240



## 9. Optional: tune a tree-based pipeline

A random forest does not need scaling, but the pipeline still handles imputation and encoding.
Notice that the search space differs and that training is slower.

In [11]:
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 6, 12],
    "model__min_samples_leaf": [1, 3, 5],
}

rf_search = GridSearchCV(
    rf_pipeline, rf_grid, cv=cv, scoring="roc_auc", n_jobs=-1
)
rf_search.fit(X_train, y_train)
print("Best RF ROC-AUC:", round(rf_search.best_score_, 4))
print("Best RF parameters:", rf_search.best_params_)

Best RF ROC-AUC: 0.9433
Best RF parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}


## 10. Leakage checklist

- Split the data before fitting any transformer.
- Put every preprocessing step inside the pipeline.
- Use `Pipeline` and `ColumnTransformer` when calling `cross_val_score` or a search.
- Do not use the test set to choose hyperparameters or thresholds.
- For time series, use `TimeSeriesSplit` rather than random folds.
- For grouped data (for example, multiple records per patient), use `GroupKFold`.

## Key takeaways

- Pipelines make preprocessing reproducible and leakage-resistant.
- `ColumnTransformer` handles mixed column types cleanly.
- `GridSearchCV` is exhaustive; `RandomizedSearchCV` scales better to large search spaces.
- Report the test score only after tuning is finished.
- Nested CV gives a less biased estimate when tuning is part of the model-selection process.

## Exercises

1. Add a `FunctionTransformer` that creates a ratio of two numeric columns and include it in the pipeline.
2. Tune the random forest with `RandomizedSearchCV` and compare the best score and runtime.
3. Implement nested cross-validation with `cross_val_score` over a `GridSearchCV` object.
4. Add an `OrdinalEncoder` for `contract` instead of one-hot encoding. Does it change the score?
5. Save the best pipeline with `joblib` and verify that predictions match after loading.